# Generate Modeling Dataset

**Purpose**: Create machine learning dataset by orchestrating existing Python scripts.


## Prerequisites

1. ✅ Run `HSA_FINAL.ipynb` to generate HSA boundaries
2. ✅ Run `Population_Allocation_Probabilistic_v2.ipynb`
3. ✅ Run `GEE_local_HSA_Weekly_Climate_Lagged.ipynb`
4. ✅ Make sure climate CSVs are downloaded to `{OUT_DIR}/DRIVE_CLIMATE_BY_HSA_DOWNLOAD/FINAL_HSA_CLIMATE/`

In [1]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
BOUNDARY_VERSION = "v7"         # change as needed (v6, v7, v8)
NETWORK          = "SYNMODINF"        # change as needed (INF or NCD)
HSA_MODE         = "footprint"  # change as needed
# Canonical disease group: must match a General_Diagnosis value in
# data/{NETWORK}_groups_of_diagnoses.csv (e.g. "Diarrheal Diseases" for INF,
# "Hypertension" for NCD). A keyword/slug (e.g. "diarrheal") also resolves, but
# the exact group name is unambiguous and is what HEFTE compares on.
DISEASE_FOCUS    = "Diarrheal Diseases"

# Date ranges
WEEK_START    = "2019-01-07"    # start of weekly aggregation window (patient data)
WEEK_END      = "2024-01-29"    # end of weekly aggregation window
ML_START_DATE = "2022-06-27"    # start of valid ML data (post diagnostic code change)
ML_END_DATE   = "2024-01-29"    # end of valid ML data
# ─────────────────────────────────────────────────────────────────────────────


In [2]:
# ============================================================================
# Derived settings and paths (do not edit below this line)
# ============================================================================
from pathlib import Path
import os
import subprocess
import sys

# --- Data Quality Thresholds ---
MISSING_THRESHOLD = 0.40               # Drop features with >40% missing data
CORRELATION_THRESHOLD = 0.95           # Remove features with r > 0.95

# --- Directory Paths ---
DATA_DIR = Path(os.environ.get("HSA_DATA_DIR", "data"))
OUT_DIR = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out")))
CLIMATE_DIR = OUT_DIR / f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_{BOUNDARY_VERSION.upper()}" / "FINAL_HSA_CLIMATE"
MODELING_DIR = OUT_DIR / "modeling"
MODELING_DIR.mkdir(exist_ok=True, parents=True)

print("="*80)
print("MODELING DATASET GENERATION - CONFIGURATION")
print("="*80)
print(f"Network:            {NETWORK}")
print(f"HSA Mode:           {HSA_MODE}")
print(f"Disease Focus:      {DISEASE_FOCUS}")
print(f"Boundary version:   {BOUNDARY_VERSION}")
print(f"Weekly aggregation: {WEEK_START} to {WEEK_END}")
print(f"ML dataset range:   {ML_START_DATE} to {ML_END_DATE}")
print(f"Missing threshold:  {MISSING_THRESHOLD*100:.0f}%")
print(f"Correlation:        {CORRELATION_THRESHOLD}")
print("="*80)


MODELING DATASET GENERATION - CONFIGURATION
Network:            SYNMODINF
HSA Mode:           footprint
Disease Focus:      Diarrheal Diseases
Boundary version:   v7
Weekly aggregation: 2019-01-07 to 2024-01-29
ML dataset range:   2022-06-27 to 2024-01-29
Missing threshold:  40%
Correlation:        0.95


## Step 1: Check Prerequisites

In [3]:
print("\nChecking prerequisites...")

# Patient data: try real data first, then synthetic variants
_patient_candidates = [
    Path("data") / f"{NETWORK}_patient_visits.csv",
    Path("data") / f"SYN{NETWORK}_patient_visits.csv",
    Path("data") / f"SYNMOD{NETWORK}_patient_visits.csv",
]
_patient_file = next((p for p in _patient_candidates if p.exists()), None)

required_files = {
    "HSA boundaries": OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson",
    "Pixel allocations": OUT_DIR / f"pixel_allocations_{NETWORK}_{HSA_MODE}_{BOUNDARY_VERSION}.csv",
    "Patient data": _patient_file or _patient_candidates[0],
    "Climate directory": OUT_DIR / f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_{BOUNDARY_VERSION.upper()}" / "FINAL_HSA_CLIMATE"
}

all_found = True
for name, filepath in required_files.items():
    if filepath.exists():
        if filepath.is_dir():
            num_files = len(list(filepath.glob("*.csv")))
            print(f"  ✓ {name}: {num_files} CSV files")
        else:
            size_mb = filepath.stat().st_size / (1024*1024)
            print(f"  ✓ {name}: {size_mb:.1f} MB")
    else:
        print(f"  ✗ {name}: NOT FOUND")
        all_found = False

if not all_found:
    raise FileNotFoundError("Missing required files. Complete prerequisite steps.")

print("\n✓ All prerequisites found!")



Checking prerequisites...
  ✓ HSA boundaries: 0.2 MB
  ✓ Pixel allocations: 744.3 MB
  ✓ Patient data: 8.4 MB
  ✓ Climate directory: 114 CSV files

✓ All prerequisites found!


## Step 2: Generate Weekly Disease Counts

Calls existing script to aggregate patient visits by HSA and week.

In [4]:
print("\n" + "="*80)
print("GENERATING WEEKLY DISEASE COUNTS")
print("="*80)

hsa_geojson = str(OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson")

# Check for allocation details (try both naming conventions)
allocation_details = OUT_DIR / f"{NETWORK}_{HSA_MODE}_allocation_details_{BOUNDARY_VERSION}.csv"
allocation_details_alt = OUT_DIR / f"pixel_allocations_{NETWORK}_{HSA_MODE}_{BOUNDARY_VERSION}.csv"
has_allocation = allocation_details.exists() or allocation_details_alt.exists()

# Try adjusted script first, fallback to regular
if Path("generate_weekly_disease_counts_adjusted.py").exists() and has_allocation:
    script = "generate_weekly_disease_counts_adjusted.py"
    which_file = allocation_details if allocation_details.exists() else allocation_details_alt
    print(f"Using adjusted counts (found {which_file})")
elif Path("generate_weekly_disease_counts.py").exists():
    if Path("generate_weekly_disease_counts_adjusted.py").exists():
        print(f"Adjusted allocation file missing; falling back to unadjusted counts.")
    script = "generate_weekly_disease_counts.py"
else:
    raise FileNotFoundError("Weekly disease counts script not found")

print(f"\nUsing: {script}")
print(f"  Week range: {WEEK_START} to {WEEK_END}\n")

result = subprocess.run(
    [sys.executable, script, hsa_geojson,
     "--out-dir", str(OUT_DIR),
     "--disease", DISEASE_FOCUS,
     "--week-start", WEEK_START,
     "--week-end", WEEK_END,
     "--boundary-version", BOUNDARY_VERSION],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code: {result.returncode}")

print("\n✓ Weekly disease counts generated!")


GENERATING WEEKLY DISEASE COUNTS
Using adjusted counts (found out/SYNMODINF_footprint_allocation_details_v7.csv)

Using: generate_weekly_disease_counts_adjusted.py
  Week range: 2019-01-07 to 2024-01-29

GENERATING ADJUSTED WEEKLY DISEASE COUNTS
Using gravity model to avoid double-counting facilities in multiple HSAs
Network: SYNMODINF | HSA mode: footprint
Disease focus: diarrheal_diseases | Secondary: synmodinf

[1/7] Loading HSA geometries...
  Loaded 19 HSAs
  Using 'healthfacility' as HSA identifier

[2/7] Loading gravity model allocations...
  Using: out/SYNMODINF_footprint_allocation_details_v7.csv
  Loaded 7,530,641 population pixels with facility assignments
  Using facilities: data/SYNMODINF_facility_coordinates.csv

[3/7] Calculating facility-to-HSA probabilities...
  Processing 19 anchor facilities (direct mapping)
  Processing 169 non-anchor facilities (using gravity model)
  Mapped 188 facilities to HSAs (19 anchors + 169 non-anchors)

[4/7] Generating Monday-anchored we

## Step 3: Merge Climate Data with Disease Counts

Calls `prepare_ml_dataset.py` to merge climate and disease data.

In [5]:
print("\n" + "="*80)
print("MERGING CLIMATE DATA WITH DISEASE COUNTS")
print("="*80)

if not Path("prepare_ml_dataset.py").exists():
    raise FileNotFoundError("prepare_ml_dataset.py not found")

print("\nUsing: prepare_ml_dataset.py")
print(f"  Climate dir: {CLIMATE_DIR}")
print(f"  Output dir: {MODELING_DIR}")
print(f"  Date range: {ML_START_DATE} to {ML_END_DATE}")
print(f"  Missing threshold: {MISSING_THRESHOLD}")
print(f"  Correlation threshold: {CORRELATION_THRESHOLD}\n")

result = subprocess.run(
    [sys.executable, "prepare_ml_dataset.py",
     "--network", NETWORK,
     "--hsa-mode", HSA_MODE,
     "--disease-focus", DISEASE_FOCUS,
     "--out-dir", str(OUT_DIR),
     "--climate-dir", str(CLIMATE_DIR),
     "--output-dir", str(MODELING_DIR),
     "--start-date", ML_START_DATE,
     "--end-date", ML_END_DATE,
     "--missing-threshold", str(MISSING_THRESHOLD),
     "--correlation-threshold", str(CORRELATION_THRESHOLD),
     "--boundary-version", BOUNDARY_VERSION],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code: {result.returncode}")

print("\n✓ ML dataset prepared!")


MERGING CLIMATE DATA WITH DISEASE COUNTS

Using: prepare_ml_dataset.py
  Climate dir: out/DRIVE_CLIMATE_BY_HSA_DOWNLOAD_V7/FINAL_HSA_CLIMATE
  Output dir: out/modeling
  Date range: 2022-06-27 to 2024-01-29
  Missing threshold: 0.4
  Correlation threshold: 0.95

ML DATASET PREPARATION
Date: 2026-10-19 00:12:33
Climate directory: out/DRIVE_CLIMATE_BY_HSA_DOWNLOAD_V7/FINAL_HSA_CLIMATE
Network: SYNMODINF
HSA mode: footprint
Disease focus: diarrheal_diseases
Diagnosis file: out/SYNMODINF_footprint_weekly_diarrheal_diseases_adjusted_v7.csv
Output directory: out/modeling

[STEP 1] Identifying HSAs...
  Found 19 HSAs
     1. AL-Nadeem_Hospital
     2. AL-Ramtha_Hospital
     3. AL-Shuneh_Hospital
     4. AL-Zarqa_Hospital
     5. Al-Basheer_Hospital
     6. Al-Karak_Hospital
     7. Al-Mafraq_Gynecology_and_Pediatrics_Hospital
     8. Al-Yarmouk_Hospital
     9. Al_Hussain_New_Salt_Hospital
    10. Dr._Jamel_Al-Totanji_Hospital
    11. Jadaa_Primary_Center
    12. Jarash_Hospital
    13. Kha

## Step 4: Verify Outputs

In [6]:
import pandas as pd
import json

print("\n" + "="*80)
print("VERIFYING OUTPUTS")
print("="*80)

# Find output files
output_patterns = [
    f"*{NETWORK}*{HSA_MODE}*weekly*.csv",
    f"*{NETWORK}*{HSA_MODE}*modeling*.csv",
    f"*{NETWORK}*{HSA_MODE}*metadata*.json"
]

found_files = []
for pattern in output_patterns:
    found_files.extend(OUT_DIR.rglob(pattern))

print("\nGenerated files:")
for filepath in sorted(set(found_files)):
    if filepath.suffix == '.csv':
        df = pd.read_csv(filepath)
        size_mb = filepath.stat().st_size / (1024*1024)
        print(f"  ✓ {filepath.name} ({len(df):,} rows, {size_mb:.2f} MB)")
    else:
        print(f"  ✓ {filepath.name}")

# Load main dataset if found
dataset_files = [f for f in found_files if 'modeling_dataset' in f.name and f.suffix == '.csv']
if dataset_files:
    print("\nDataset Summary:")
    df = pd.read_csv(dataset_files[0])
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
    if 'hsa_id' in df.columns:
        print(f"  HSAs: {df['hsa_id'].nunique()}")
    if 'week_start' in df.columns:
        print(f"  Weeks: {df['week_start'].nunique()}")
        print(f"  Date range: {df['week_start'].min()} to {df['week_start'].max()}")
    if len(df.filter(regex='_count_adjusted$').columns) > 0:
        print(f"\n  Diarrheal diseases:")
        print(f"    Total cases: {df.filter(regex='_count_adjusted$').iloc[:,0].sum():,}")
        print(f"    Mean per week: {df.filter(regex='_count_adjusted$').iloc[:,0].mean():.2f}")
        print(f"    Weeks with cases: {(df.filter(regex='_count_adjusted$').iloc[:,0] > 0).sum():,}")

print("\n" + "="*80)
print("COMPLETE")
print("="*80)
print("\nNext: Use dataset in modeling pipeline")


VERIFYING OUTPUTS

Generated files:
  ✓ SYNMODINF_footprint_weekly_diarrheal_diseases_adjusted_v7.csv (5,035 rows, 0.24 MB)
  ✓ SYNMODINF_footprint_weekly_synmodinf_adjusted_v7.csv (5,035 rows, 0.24 MB)
  ✓ SYNMODINF_footprint_modeling_dataset_v7.csv (1,596 rows, 3.44 MB)
  ✓ SYNMODINF_footprint_modeling_dataset_v7_metadata.json

Dataset Summary:
  Rows: 1,596
  Columns: 153
  HSAs: 19
  Weeks: 84
  Date range: 2022-06-27 to 2024-01-29

  Diarrheal diseases:
    Total cases: 18,432.0
    Mean per week: 11.55
    Weeks with cases: 1,374

COMPLETE

Next: Use dataset in modeling pipeline


In [7]:
print("\n" + "="*80)
print("PACKAGING RESULTS ARCHIVE")
print("="*80)

# package_results.py builds a network- and disease-scoped archive name:
#   hsa_results_<NETWORK>_<HSA_MODE>_<disease_slug>_<BOUNDARY_VERSION>.zip
# and filters the shared climate download folder to this network's files, so
# INF and NCD never land in the same archive. Passing --disease-focus also
# writes the canonical disease group (from groups_of_diagnoses) into manifest.json.
result = subprocess.run(
    [sys.executable, "package_results.py",
     "--network",       NETWORK,
     "--mode",          HSA_MODE,
     "--version",       BOUNDARY_VERSION,
     "--disease-focus", DISEASE_FOCUS],
    check=False,
)
if result.returncode != 0:
    print(f"WARNING: package_results.py exited with code {result.returncode}")
else:
    print("\n✓ Results archive ready (see 'Wrote ...' above).")



PACKAGING RESULTS ARCHIVE


usage: package_results.py [-h] [--network NETWORK] [--mode MODE]
                          [--version VERSION] [--output OUTPUT]
package_results.py: error: unrecognized arguments: --disease-focus Diarrheal Diseases


In [8]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "Generate_Modeling_Dataset"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', "out")))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', os.environ.get('PIPELINE_VERSION', 'v7')))
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_{BOUNDARY_VERSION}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet', '.zip'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

# The executed-cell-output section is appended by run_pipeline.py after the
# kernel exits. It cannot be produced here: at this point the current run's
# outputs exist only in kernel memory, and the .ipynb on disk still holds the
# previous run's, which is what this cell used to copy into the summary.

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")


Saved notebook summary: out/textresults/Generate_Modeling_Dataset_SYNMODINF_footprint_v7_results.md
